In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [20]:
df = pd.read_excel(f'pertussis1year.xlsx')
df['CREATE_DATE'] = df['CREATE_DATE'].astype('datetime64[ns]')

/opt/homebrew/lib/python3.13/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [ ]:
df['CREATE_DATE'].describe()

count                             2093
mean     2022-07-11 07:26:31.017678080
min                2019-11-01 00:00:00
25%                2021-01-21 00:00:00
50%                2022-09-06 00:00:00
75%                2024-01-04 00:00:00
max                2024-11-01 00:00:00
Name: CREATE_DATE, dtype: object

In [ ]:
df.set_index('CREATE_DATE',inplace=True)
df['CASE_COUNT'] = 1
df = df[['CASE_COUNT']]
df_resampled = df.resample('W').sum()

In [ ]:

df.columns
df[['AGE']].hist()
sns.kdeplot(df[['AGE']])
df['AGE'].describe()
df['CREATE_DATE'].astype('datetime64[ns]').hist()

df_resampled = df_resampled.drop('2024-11-30') # November isn't over, provides poor estimate
df_resampled['CASE COUNT'].plot()

In [ ]:
# Assuming we can get to this point, which we just showed, but we have more data
linelist = pd.read_csv(f'Pertussis_Line_List.csv')
linelist['datetime'] = linelist['datetime'].astype('datetime64[ns]')
linelist = linelist.set_index('datetime')
linelist = linelist.resample('M').sum()
linelist.plot(figsize=(32,9))

In [ ]:
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt

# Assuming 'linelist' has been set up with a datetime index and 'cases' as the target column
result = seasonal_decompose(linelist['cases'], model='additive', period=12)  # Adjust period based on data frequency

# Plot the components
result.plot()
plt.show()

In [ ]:
import numpy as np
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.seasonal import seasonal_decompose

# Decompose to get trend, seasonality, and residuals
result = seasonal_decompose(linelist['cases'], model='additive', period=12)
trend = result.trend.dropna()  # Drop NaNs
seasonal = result.seasonal[:12]  # Seasonal component for one year

# Step 1: Fit a trend model (Linear Regression for simplicity)
X = np.arange(len(trend)).reshape(-1, 1)
y = trend.values
trend_model = LinearRegression().fit(X, y)

# Project the trend into the future (let’s say for one additional year)
future_days = 12
future_X = np.arange(len(trend), len(trend) + future_days).reshape(-1, 1)
future_trend = trend_model.predict(future_X)

# Step 2: Extend the seasonality into the future
future_seasonal = np.tile(seasonal, int(np.ceil(future_days / len(seasonal))))[:future_days]

# Combine future trend and seasonal components
future_prediction = future_trend + future_seasonal

In [ ]:
# Plot the forecast
import matplotlib.pyplot as plt
plt.plot(linelist['cases'], label='Observed')
plt.plot(pd.date_range(linelist.index[-1], periods=future_days, freq='M'),future_prediction, label='Forecast', color='orange')
plt.xlabel('Date')
plt.ylabel('Cases')
plt.legend()
plt.title('Pertussis Case Forecast')
plt.show()

In [ ]:

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt

# Decompose to get trend, seasonal, and residual components
result = seasonal_decompose(linelist['cases'], model='additive', period=12)
trend = result.trend.dropna()
seasonal = result.seasonal[:12]
residuals = result.resid.dropna()

# Step 1: Fit a trend model for forecasting
X = np.arange(len(trend)).reshape(-1, 1)
y = trend.values
trend_model = LinearRegression().fit(X, y)

# Project the trend into the future (e.g., 12 days)
future_days = 108
future_X = np.arange(len(trend), len(trend) + future_days).reshape(-1, 1)
future_trend = trend_model.predict(future_X)

# Step 2: Extend the seasonality into the future
future_seasonal = np.tile(seasonal, int(np.ceil(future_days / len(seasonal))))[:future_days]

# Combine future trend and seasonal components to get the forecast
future_prediction = future_trend + future_seasonal

# Step 3: Calculate confidence intervals
residual_std = residuals.std()  # Standard deviation of the residuals
confidence_interval = 1.96 * residual_std  # 95% confidence interval

# Upper and lower bounds
upper_bound = future_prediction + confidence_interval
lower_bound = future_prediction - confidence_interval

# Step 4: Plot the forecast with confidence bounds
plt.figure(figsize=(12, 6))
plt.plot(linelist['cases'], label='Observed')
plt.plot(pd.date_range(linelist.index[-1], periods=future_days, freq='M'),
         future_prediction, label='Forecast', color='orange')
plt.fill_between(pd.date_range(linelist.index[-1], periods=future_days, freq='M'),
                 lower_bound, upper_bound, color='orange', alpha=0.2, label='95% Confidence Interval')
plt.xlabel('Date')
plt.ylabel('Cases')
plt.legend()
plt.title('Pertussis Case Forecast with Confidence Bounds')
plt.show()
